## OS, Env vars, Path
Two completely **separate machineries** are at play here, and conflating them is the #1 source of confusion (and a fair interview trap).

---

### The big picture

- **OS `PATH`** answers: _"when I type `python`, which `python.exe` actually runs?"_
- **Python's `sys.path`** answers: _"once that interpreter is running, where does `import foo` look?"_

`Path` is a variable that contains a list of paths to directories that contain executables. `Path` is kept by your system as a way to have programs always close at hand. [Link](https://realpython.com/add-python-to-path/#understanding-what-path-is)

PATH is resolved once, by the shell, before Python even starts. `sys.path` is a Python-internal list, rebuilt fresh every time the interpreter boots. PATH influences `sys.path` only indirectly — by determining _which_ interpreter you got, and that interpreter has its own stdlib/site-packages baked into its installation directory.

> Note: A [path](https://en.wikipedia.org/wiki/Path_(computing)) is the address of a file or folder on your hard drive. The `PATH` [environment variable](https://en.wikipedia.org/wiki/Environment_variable), also referred to as just `PATH` or `Path`, is a list of paths to directories that your operating system keeps and uses to find executable scripts and programs.

1. View and Set Env Vars. [Link](https://kevinlinxc.medium.com/viewing-and-setting-environment-variables-in-every-shell-4df43127bd8#d419)
2. Which `python.exe` is actually running in your current environment is this:
```bash
Get-Command python

CommandType     Name                                               Version    Source                                                                                                                      
-----------     ----                                               -------    ------                                                                                                                      
Application     python.exe                                         3.13.11... C:\Users\Dennie\Mate Academy\Python Core\python_essentials\.venv\Scripts\python.exe   
```
This tells you **the exact path** PowerShell resolves when you type `python`.
3. `where.exe python` — Windows-native lookup
```bash
where.exe python
C:\Users\Dennie\Mate Academy\Python Core\python_essentials\.venv\Scripts\python.exe
C:\Python313\python.exe
```
Shows all python executables on `PATH`

---

## 1. Windows `PATH` — System vs User scope

In your environment:

```shell
SYSTEM scope (HKLM\SYSTEM\...\Environment) — all users, needs admin
  C:\Python314\Scripts\, C:\Python314\, C:\Python313\Scripts\, C:\Python313\, ...

USER scope (HKCU\Environment) — just your account
  c:\python313\Scripts, C:\Python314\Scripts\, ..., C:\Users\Dennie\AppData\Roaming\Python\Scripts
```

When Windows spawns a **new process** (e.g. you open a fresh PowerShell window), it builds that process's `PATH` by concatenating **System PATH first, then User PATH appended**, separated by `;`. That merged string becomes `$env:PATH` for every process descending from that shell — including `python.exe` itself.

This merge happens **only at process creation**. If you edit PATH in the Environment Variables GUI while a terminal is already open, that terminal keeps its stale copy — classic "I added it but it's not working" bug. You need a new shell.

When you type `python` at a prompt, PowerShell/cmd walks each directory in `$env:PATH` **left to right** and runs the first `python.exe` it finds. Your data: `C:\Python314\Scripts\` and `C:\Python314\` come before `C:\Python313\...` in System scope — but `(Get-Command python).Source` returned `C:\Python313\python.exe`. That tells you your **User**-scope entry `c:\python313\Scripts` (first line, lowercase) is winning, because PowerShell's actual resolution also factors in `App Execution Aliases` / `PATHEXT` and per-shell search order nuances — worth a `Get-Command -All python` if you ever want to see every candidate ranked.

> **This is exactly the mechanism a venv uses.** `venv\Scripts\activate` doesn't touch `PYTHONPATH` at all — it just **prepends the venv's `Scripts\` directory to `PATH`** for that shell session, so `python` now resolves to `venv\Scripts\python.exe` instead of your global one. Different `python.exe` → different baked-in `sys.path` → different importable packages. The whole venv illusion is a PATH trick, not an import-system trick.

---

### 2. `sys.path` — what the interpreter actually searches

Your real `sys.path` for `C:\Python313\python.exe`:

```shell
C:\Python313\python313.zip      ← compiled-in stdlib zip (often unused/empty on Windows)
C:\Python313\DLLs                ← C extension modules (.pyd)
C:\Python313\Lib                 ← pure-Python stdlib
C:\Python313                     ← interpreter's own install root
C:\Python313\Lib\site-packages   ← pip-installed third-party packages
```

Construction order at startup, in general:

1. **`sys.path[0]`** — special-cased: the directory containing the script you ran (`python script.py` → that script's folder), or `''` (meaning current working dir) for `-c`, `-m`, and the REPL. This is _why_ a script can `import` its sibling files with no setup — and it's also why a stray `random.py` sitting next to your script silently shadows the stdlib `random` module.
2. **`PYTHONPATH`** directories, if the env var is set — inserted right after entry #1.
3. The installation's baked-in paths (what you saw above) — stdlib dirs are compiled into the interpreter at build time, not discovered dynamically.
4. **`site-packages`**, appended last by the `site` module, which Python auto-imports at startup unless launched with `-S`. `site` also scans `site-packages` for `.pth` files and appends any extra paths they list (this is how `pip install -e .` / editable installs sneak your project's source dir onto `sys.path` without you setting anything manually).

---